In [0]:
WITH tb_historico AS (
    SELECT  CASE
                -- Perfil 1: Cliente novo (tenure até 12 meses)
                WHEN CAST(tenure AS INT) <= 12
                    THEN 'Perfil 1: Cliente novo (tenure até 12 meses). AÇÃO SUGERIDA: Programa de onboarding e acompanhamento.'  
                -- Perfil 2 - Risco alto: contrato mensal, fibra e pagamento não automático
                -- Migrar para: contrato anual, fibra, pagamento automático
                WHEN LOWER(Contract) = 'one year'
                    AND LOWER(InternetService) = 'fiber optic'
                    AND LOWER(PaymentMethod) LIKE '%automatic%'
                    THEN 'Perfil 2 - Risco alto: contrato mensal, fibra e pagamento não automático. AÇÃO SUGERIDA: Oferta de contrato anual com débito automático.'            
                -- Perfil 3 - Risco alto: contrato mensal, fibra, sem suporte e sem segurança
                -- Migrar para: contrato mensal, fibra, com suporte e com segurança
                WHEN LOWER(Contract) = 'month-to-month'
                    AND LOWER(InternetService) = 'fiber optic'
                    AND LOWER(OnlineSecurity) = 'yes'
                    AND LOWER(TechSupport) = 'yes'
                    THEN 'Perfil 3 - Risco alto: contrato mensal, fibra, sem suporte e sem segurança. AÇÃO SUGERIDA: Suporte técnico e segurança online gratuitos por alguns meses.'
                -- Perfil 4 - Risco médio: contrato mensal, sem fibra
                WHEN LOWER(Contract) = 'month-to-month'
                    AND LOWER(InternetService) <> 'fiber optic'
                    THEN 'Perfil 4 - Risco médio: contrato mensal, sem fibra. AÇÃO SUGERIDA: Contrato de menor custo: e-mail ou SMS com oferta de fidelidade.'  
                -- Perfil 5: Risco baixo - anual ou bienal com pagamento automático
                WHEN LOWER(Contract) <> 'month-to-month'
                    AND LOWER(PaymentMethod) LIKE '%automatic%'
                    THEN 'Perfil 5: Risco baixo - anual ou bienal com pagamento automático. Sem AÇÃO SUGERIDA.'  
                ELSE 'Outros'
            END AS perfil,
            AVG(IF(LOWER(Churn) = 'yes', 1, 0)) AS txResposta,
            MEDIAN(IF(tenure <> 'unknown', CAST(tenure AS INT), 0)) AS mediana_tenure   
    FROM    workspace.default.train_classificado
    WHERE   tenure <> 'unknown'
    GROUP BY ALL
),
tb_predicao AS (
    SELECT  *,        
            CASE
                -- Perfil 1: Cliente novo (tenure até 12 meses)
                WHEN CAST(tenure AS INT) <= 12
                    THEN 'Perfil 1: Cliente novo (tenure até 12 meses). AÇÃO SUGERIDA: Programa de onboarding e acompanhamento.'  
                -- Perfil 2 - Risco alto: contrato mensal, fibra e pagamento não automático
                WHEN LOWER(Contract) = 'month-to-month'
                    AND LOWER(InternetService) = 'fiber optic'
                    AND LOWER(PaymentMethod) NOT LIKE '%automatic%'
                    THEN 'Perfil 2 - Risco alto: contrato mensal, fibra e pagamento não automático. AÇÃO SUGERIDA: Oferta de contrato anual com débito automático.'            
                -- Perfil 3 - Risco alto: contrato mensal, fibra, sem suporte e sem segurança
                WHEN LOWER(Contract) = 'month-to-month'
                    AND LOWER(InternetService) = 'fiber optic'
                    AND LOWER(OnlineSecurity) = 'no'
                    AND LOWER(TechSupport) = 'no'
                    THEN 'Perfil 3 - Risco alto: contrato mensal, fibra, sem suporte e sem segurança. AÇÃO SUGERIDA: Suporte técnico e segurança online gratuitos por alguns meses.'
                -- Perfil 4 - Risco médio: contrato mensal, sem fibra
                WHEN LOWER(Contract) = 'month-to-month'
                    AND LOWER(InternetService) <> 'fiber optic'
                    THEN 'Perfil 4 - Risco médio: contrato mensal, sem fibra. AÇÃO SUGERIDA: Contrworkspace.default.inference_classificado_predicaoato de menor custo: e-mail ou SMS com oferta de fidelidade.'  
                -- Perfil 5: Risco baixo - anual ou bienal com pagamento automático
                WHEN LOWER(Contract) <> 'month-to-month'
                    AND LOWER(PaymentMethod) LIKE '%automatic%'
                    THEN 'Perfil 5: Risco baixo - anual ou bienal com pagamento automático. Sem AÇÃO SUGERIDA.'  
                ELSE 'Outros'
            END AS perfil
    FROM    inference_classificado_predicao
    WHERE   tenure <> 'unknown'
)
SELECT  pred.*,
        hist.mediana_tenure,
        pred.probChurn 
            * MonthlyCharges 
            * IF(
                pred.perfil ILIKE 'Perfil 1%' OR pred.perfil ILIKE 'Perfil 4%', 3,
                IF(
                    pred.perfil = 'Outros' OR pred.perfil ILIKE 'Perfil 5%', NULL,
                    IF(hist.mediana_tenure - CAST(tenure AS INT) < 3, 3, hist.mediana_tenure - CAST(tenure AS INT))
                )
        ) AS vlExpectativaReceitaNaoChurn,
        IF(
             pred.perfil ILIKE 'Perfil 1%' OR pred.perfil ILIKE 'Perfil 4%', 3,
             IF(
                 pred.perfil = 'Outros' OR pred.perfil ILIKE 'Perfil 5%', NULL,
                 IF(hist.mediana_tenure - CAST(tenure AS INT) < 3, 3, hist.mediana_tenure - CAST(tenure AS INT))
             )
        ) AS qtdMesesExpectatuvaNaoChurn
FROM    tb_predicao AS pred
        LEFT JOIN tb_historico AS hist
            ON pred.perfil = hist.perfil
ORDER BY pred.probChurn DESC

In [0]:
SELECT  motivo_feedback,
        churnPredicao,
        COUNT(*) AS qtd
FROM    workspace.default.inference_classificado_predicao
GROUP BY ALL
ORDER BY qtd DESC

Databricks visualization. Run in Databricks to view.